# Objetivo
Desenvolver biblioteca para cálculo de ofensores de chamadas abusivas. 

Neste notebook serão desenvolvidas, sobre uma amostra dos dados de CDRs, funções para:

- extrair CDRs processados do formato texto para parquet;
- transformar CDRs para formato normalizado (tabela única com campos uniformes);
- carregar e calcular chamadas curtas;

> Atenção: este notebook deve ser executado no kernel com Python 3.9

# Configuração do ambiente

## Bibliotecas

In [ ]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T

from teleutils.preprocessing import normalize_number

## Variáveis de ambiente

In [ ]:
LIMIAR_CHAMADA_OFENSORA = 6

In [ ]:
TRANSFORMED_COLUMNS = [
    "referencia", 
    "tipo_de_chamada", 
    "data_hora", 
    "numero_de_a_formatado", 
    "numero_de_b_formatado", 
    "hora_da_chamada",
    "duracao_da_chamada", 
    "chamada_curta", 
    "chamada_autenticada", 
    "chamada_caixa_postal",
]

## Funções auxiliares

In [ ]:
# Definir schema
return_schema = T.StructType([
    T.StructField("numero_formatado", T.StringType(), True),
    T.StructField("numero_valido", T.BooleanType(), True),
])

@pandas_udf(return_schema)
def spark_normalize_number(number_series: pd.Series) -> pd.DataFrame:
    # Processar em batch (vetorizado)
    results = []
    for number in number_series:
        results.append(normalize_number(number))
    
    return pd.DataFrame(results, columns=[
        "numero_formatado", "numero_valido",
    ])

## Spark Session

In [ ]:
spark = SparkSession.builder \
    .master("local[1]") \
    .appName("desenvolvimento_chamadas_abusivas") \
    .config("spark.executor.memory", "512m") \
    .config("spark.ui.port", "8890") \
    .getOrCreate()
spark

## Pastas origem e destino

In [ ]:
# Pasta base origem
INPUT_FOLDER = "/data/cdr/chamadas_abusivas/cdr_processado/Semana86"

# Pasta de arquivos extraídos
EXTRACTED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_extraido/Semana86"

# Pasta de arquivos transformados
TRANSFORMED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_transformado/Semana86"

## Caminho completo dos arquivos

In [ ]:
# Claro/Ericsson
cdr_processado_claro_ericsson = f"{INPUT_FOLDER}/CLARO*Ericsson"
cdr_extraido_claro_ericsson = f"{EXTRACTED_FOLDER}/claro_ericsson_extraido.parquet"
cdr_transformado_claro_ericsson = f"{TRANSFORMED_FOLDER}/claro_ericsson_transformado.parquet"

# Tim/Ericsson

# Vivo/Ericsson

# Tim/Volte
cdr_processado_tim_volte = f"{INPUT_FOLDER}/TIM*Volte"
cdr_extraido_tim_volte = f"{EXTRACTED_FOLDER}/tim_volte_extraido.parquet"
cdr_transformado_tim_volte = f"{TRANSFORMED_FOLDER}/tim_volte_transformado.parquet"

# Tim/Stir
cdr_processado_tim_stir = f"{INPUT_FOLDER}/TIM*Stir"
cdr_extraido_tim_stir = f"{EXTRACTED_FOLDER}/tim_stir_extraido.parquet"
cdr_transformado_tim_stir = f"{TRANSFORMED_FOLDER}/tim_stir_transformado.parquet"

# Tim/Volte
cdr_processado_vivo_volte = f"{INPUT_FOLDER}/VivoVolteConsolidado"
cdr_extraido_vivo_volte = f"{EXTRACTED_FOLDER}/vivo_volte_extraido.parquet"
cdr_transformado_vivo_volte = f"{TRANSFORMED_FOLDER}/vivo_volte_transformado.parquet"

# Desenvolvimento

## Extração

### Ericsson

In [ ]:
%%time
source_file = cdr_processado_claro_ericsson
target_file = cdr_extraido_claro_ericsson

# Obter o SparkContext
sc = spark.sparkContext
sc.setJobDescription("Lendo arquivo CSV")

df = (
    spark.read
    .format("csv")
    .option("delimiter", ";")
    .option("header", True)
    .load(source_file)
)

# Selecionar e renomear as colunas
sc.setJobDescription("Selecionando e renomeando as colunas")
usecols = [0, 1, 2, 3, 4, 9, 11]
names = ["referencia", "numero_de_a", "_data", "_hora", "tipo_de_chamada", "numero_de_b", "duracao_da_chamada"]
columns_to_keep = [df.columns[i] for i in usecols]
df = df.select(columns_to_keep).toDF(*names)

# Gravar particionado por tipo_de_chamada
# As partições não contém as colunas utilizadas para particionamento
sc.setJobDescription("Gravando arquivo Parquet particionado")
df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)

In [ ]:
df = spark.read.parquet(target_file)
df.show(5)

### Tim/Volte

In [ ]:
%%time
source_file = cdr_processado_tim_volte
target_file = cdr_extraido_tim_volte
# Obter o SparkContext
sc = spark.sparkContext

# Ler arquivos .txt.gz
sc.setJobDescription("Lendo arquivo CSV Tim/Volte")
df = (
    spark.read
    .format("csv")
    .option("delimiter", ";")
    .option("header", True)
    .load(source_file)
)

# Selecionar e renomear as colunas
sc.setJobDescription("Selecionando e renomeando as colunas")
usecols = [0, 1, 2, 3, 4, 7, 12]
names = ["numero_de_a", "_data", "_hora", "tipo_de_chamada", "numero_de_b", "duracao_da_chamada", "referencia"]
columns_to_keep = [df.columns[i] for i in usecols]
df = df.select(columns_to_keep).toDF(*names)

# Gravar particionado por tipo_de_chamada
# As partições não contém as colunas utilizadas para particionamento
sc.setJobDescription("Gravando arquivo Parquet particionado")
df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)

df = spark.read.parquet(target_file)
df.show(5)

### Tim/Stir

In [ ]:
%%time
source_file = cdr_processado_tim_stir
target_file = cdr_extraido_tim_stir

# Obter o SparkContext
sc = spark.sparkContext

# Ler arquivos .txt.gz
sc.setJobDescription("Lendo arquivo CSV Tim/Stir")
df = spark.read.format("csv").option("delimiter", ";").option("header", True).load(source_file)

# Selecionar e renomear as colunas
sc.setJobDescription("Selecionando e renomeando as colunas")
usecols = [0, 1, 2, 5, 6, 11, 13, 14]
names = ["numero_de_a", "_data", "_hora", "tipo_de_chamada", "referencia", "numero_de_b", "duracao_da_chamada", "autenticacao"]
columns_to_keep = [df.columns[i] for i in usecols]
df = df.select(columns_to_keep).toDF(*names)

# Gravar particionado por tipo_de_chamada
sc.setJobDescription("Gravando arquivo Parquet particionado")
df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)

df = spark.read.parquet(target_file)
df.show(5)

### Vivo/Volte

In [ ]:
%%time
source_file = cdr_processado_vivo_volte
target_file = cdr_extraido_vivo_volte

# Ler arquivos .txt.gz
sc.setJobDescription("Lendo arquivo CSV")
df = (
    spark.read
    .format("csv")
    .option("delimiter", "|")
    .option("header", False)
    .load(source_file)
)

# Selecionar e renomear as colunas
sc.setJobDescription("Selecionando e renomeando as colunas")
usecols=[0,2,5,12,13,31,45]
names = ["tipo_de_chamada", "numero_de_a", "numero_de_b", "duracao_da_chamada", "_data", "_hora", "referencia"]
columns_to_keep = [df.columns[i] for i in usecols]
df = df.select(columns_to_keep).toDF(*names)

# Gravar particionado por tipo_de_chamada
# As partições não contém as colunas utilizadas para particionamento
sc.setJobDescription("Gravando arquivo Parquet particionado")
df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)

df = spark.read.parquet(target_file)
df.show(5)

## Transformação

In [ ]:
_return_schema = T.StructType(
    [
        T.StructField("numero_formatado", T.StringType(), True),
        T.StructField("numero_valido", T.BooleanType(), True),
    ]
)


@pandas_udf(_return_schema)
def _spark_normalize_number(number_series: pd.Series) -> pd.DataFrame:
    # Processar em batch (vetorizado)
    results = [normalize_number(n) for n in number_series]

    return pd.DataFrame(
        results,
        columns=[
            "numero_formatado",
            "numero_valido",
        ],
    )

In [ ]:
def _format_columns(df, date_time_fmt: str = "yyyy-MM-dd HH-mm-ss"):
    return (
        df.withColumn(
            "duracao_da_chamada",
            F.coalesce(F.col("duracao_da_chamada").cast(T.IntegerType()), F.lit(0)),
        )
        .withColumn(
            "data_hora",
            F.to_timestamp(F.concat_ws(" ", F.col("_data"), F.col("_hora")), date_time_fmt),
        )
        .withColumn("hora_da_chamada", F.date_format(F.col("data_hora"), "yyyyMMddHH"))
    )

In [ ]:
def _format_numbers(df):
    return (
        df.withColumn(
            "numero_de_a_formatado",
            spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
        .withColumn( 
            "numero_de_b_formatado",
            spark_normalize_number("numero_de_b").getField("numero_formatado")
        )
    )

In [ ]:
def _add_chamada_curta(df):
    return df.withColumn(
        "chamada_curta",
        F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1)
         .otherwise(F.lit(0)),
    )

### Ericsson

In [ ]:
source_file = cdr_extraido_claro_ericsson
target_file = cdr_transformado_claro_ericsson
date_time_fmt = "yyyy-MM-dd HH:mm:ss"

# ler chamadas terminadas
df = spark.read.parquet(source_file).filter(F.col("tipo_de_chamada") == "TER")
df = _format_columns(df, date_time_fmt)
df = _format_numbers(df)
df = _add_chamada_curta(df)
df = df.withColumn("chamada_autenticada",F.lit(0)).withColumn("chamada_caixa_postal",F.lit(0)).select(TRANSFORMED_COLUMNS)
df.show(5)

In [ ]:
df = df.withColumn(
    "chamada_curta",
    F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1).otherwise(F.lit(0))
).withColumn("chamada_autenticada",F.lit(0)).withColumn("caixa_postal",F.lit(0))
df.select(TRANSFORMED_COLUMNS).show(5)

In [ ]:
%%time
source_file = cdr_extraido_claro_ericsson
target_file = cdr_transformado_claro_ericsson

df = (
    spark.read
        .parquet(source_file)
        .filter(F.col("tipo_de_chamada") == "TER")
        .withColumn(
            "data_hora",
            F.to_timestamp(F.concat_ws(" ", F.col("_data"), F.col("_hora"))))
        .withColumn("hora_da_chamada",F.date_format(F.col("data_hora"),"yyyyMMddHH"))
        .withColumn("duracao_da_chamada",F.col("duracao_da_chamada").cast(T.IntegerType()))
        .withColumn(
            "chamada_curta",
            F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1).otherwise(F.lit(0))
        )
        .withColumn(
            "numero_de_a_formatado",
            spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
        .withColumn( 
            "numero_de_b_formatado",
            spark_normalize_number("numero_de_b").getField("numero_formatado")
        )
        .withColumn("chamada_autenticada",F.lit(0))
        .withColumn("caixa_postal",F.lit(0))
        .select(TRANSFORMED_COLUMNS)
)
df.write.mode("overwrite").parquet(target_file)
df.show(5)

### Tim/Volte

In [ ]:
%%time
source_file = cdr_extraido_tim_volte
target_file = cdr_transformado_tim_volte

In [ ]:
source_file = cdr_extraido_tim_volte
target_file = cdr_transformado_tim_volte
date_time_fmt = "yyyy-MM-dd HH-mm-ss"

# ler caixa postal
df_caixa_postal = (
    spark.read.parquet(source_file)
    .filter(F.col("tipo_de_chamada") == "FORv")
    .filter(
        (F.col("numero_de_b").startswith("5505")) &
        (F.length("numero_de_b") == 6)
    )
    .select("referencia")
    .distinct()
    .withColumn("chamada_caixa_postal", F.lit(1))
)

# ler chamadas terminadas
df = spark.read.parquet(source_file).filter(F.col("tipo_de_chamada") == "TERv")

df = _format_columns(df, date_time_fmt)
df = _format_numbers(df)
df = _add_chamada_curta(df)
df = df.join(
    df_caixa_postal,
    on="referencia",
    how="left"
).withColumn(
    "chamada_caixa_postal", 
    F.coalesce(F.col("chamada_caixa_postal"), F.lit(0))
).withColumn("chamada_autenticada",F.lit(0)).select(TRANSFORMED_COLUMNS)
df.show(5)

In [ ]:
%%time
source_file = cdr_extraido_tim_volte
target_file = cdr_transformado_tim_volte

df_caixa_postal = (
    spark.read.parquet(source_file)
    .filter(F.col("tipo_de_chamada") == "FORv")
    .filter(
        (F.col("numero_de_b").startswith("5505")) &
        (F.length("numero_de_b") == 6)
    )
    .select("referencia")
    .distinct()
    .withColumn("caixa_postal", F.lit(1))
    .cache()
)


df = (
    spark.read
        .parquet(source_file)
        .filter(F.col("tipo_de_chamada") == "TERv")
        .withColumn(
            "data_hora",
            F.to_timestamp(F.concat_ws(" ", F.col("_data"), F.col("_hora")), "yyyy-MM-dd HH-mm-ss"))
        .withColumn(
            "duracao_da_chamada", 
            F.coalesce(F.col("duracao_da_chamada").cast(T.IntegerType()), F.lit(0))
        )
        .groupBy("referencia", "tipo_de_chamada", "numero_de_a", "numero_de_b")
        .agg(
            F.min("data_hora").alias("data_hora"),
            F.sum("duracao_da_chamada").alias("duracao_da_chamada")
        )
        .withColumn("hora_da_chamada",F.date_format(F.col("data_hora"),"yyyyMMddHH"))
        .withColumn(
            "chamada_curta",
            F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1).otherwise(F.lit(0))
        )
        .withColumn(
            "numero_de_a_formatado",
            spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
        .withColumn( 
            "numero_de_b_formatado",
            spark_normalize_number("numero_de_b").getField("numero_formatado")
        )
        .withColumn("chamada_autenticada",F.lit(0))
        .join(
            F.broadcast(df_caixa_postal),
            on="referencia",
            how="left"
        )
        .fillna(0, subset=["caixa_postal"])
        .withColumn("caixa_postal", F.coalesce(F.col("caixa_postal"), F.lit(0)))
        .select(TRANSFORMED_COLUMNS)
)
df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)
df.show(5)

In [ ]:
df = (
    spark.read
        .parquet(source_file)
        .filter(F.col("tipo_de_chamada") == "TERv")
        .withColumn(
            "data_hora",
            F.to_timestamp(F.concat_ws(" ", F.col("_data"), F.col("_hora")), "yyyy-MM-dd HH-mm-ss"))
        .withColumn(
            "duracao_da_chamada", 
            F.coalesce(F.col("duracao_da_chamada").cast(T.IntegerType()), F.lit(0))
        )
        .groupBy("referencia", "tipo_de_chamada", "numero_de_a", "numero_de_b")
        .agg(
            F.min("data_hora").alias("data_hora"),
            F.sum("duracao_da_chamada").alias("duracao_da_chamada")
        )
        .withColumn("hora_da_chamada",F.date_format(F.col("data_hora"),"yyyyMMddHH"))
        .withColumn(
            "chamada_curta",
            F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1).otherwise(F.lit(0))
        )
        .withColumn(
            "numero_de_a_formatado",
            spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
        .withColumn( 
            "numero_de_b_formatado",
            spark_normalize_number("numero_de_b").getField("numero_formatado")
        )
        .withColumn("caixa_postal",F.lit(0))
)
df.show(5)

In [ ]:
df.join(F.broadcast(df_caixa_postal),on="referencia",how="left").show(5)

### Tim/Stir

In [ ]:
source_file = cdr_extraido_tim_stir
target_file = cdr_transformado_tim_volte
date_time_fmt = "yyyy-MM-dd HH-mm-ss"
sip_number_pattern = r"sip:(\d+)@"

# ler chamadas terminadas
df = spark.read.parquet(source_file).filter(F.col("tipo_de_chamada") == "82")

df = _format_columns(df, date_time_fmt)
df = df.withColumn("numero_de_b", F.regexp_extract(F.col("numero_de_b"), sip_number_pattern, 1))
df = _format_numbers(df)
df = _add_chamada_curta(df)
df = df.withColumn(
        "chamada_autenticada",
        F.when(F.col("autenticacao").startswith("TN-Validation-Pa"), 1).otherwise(F.lit(0))
    ).withColumn("chamada_caixa_postal",F.lit(0)).select(TRANSFORMED_COLUMNS)
df.show(5)

In [ ]:
%%time
source_file = cdr_extraido_tim_stir
target_file = cdr_transformado_tim_stir

sip_number_pattern = r"sip:(\d+)@"

df = (
    spark.read
    .parquet(source_file)
    .filter(F.col("tipo_de_chamada") == "82")
    .withColumn("numero_de_b", F.regexp_extract(F.col("numero_de_b"), sip_number_pattern, 1))
    .withColumn(
            "data_hora",
            F.to_timestamp(F.concat_ws(" ", F.col("_data"), F.col("_hora")), "yyyy-MM-dd HH-mm-ss"))
    .withColumn(
            "duracao_da_chamada", 
            F.coalesce(F.col("duracao_da_chamada").cast(T.IntegerType()), F.lit(0))
        )
    .groupBy("referencia", "tipo_de_chamada", "numero_de_a", "numero_de_b", "autenticacao")
    .agg(
        F.min("data_hora").alias("data_hora"),
        F.sum("duracao_da_chamada").alias("duracao_da_chamada")
    )
    .withColumn(
        "numero_de_a_formatado",
        spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
    .withColumn( 
        "numero_de_b_formatado",
        spark_normalize_number("numero_de_b").getField("numero_formatado")
    )
    .withColumn("hora_da_chamada",F.date_format(F.col("data_hora"),"yyyyMMddHH"))
    .withColumn(
        "chamada_autenticada",
        F.when(F.col("autenticacao").startswith("TN-Validation-Pa"), 1).otherwise(F.lit(0))
    )
    .withColumn(
        "chamada_curta",
        F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1).otherwise(F.lit(0))
    )
    .withColumn("caixa_postal",F.lit(0))
    .select(TRANSFORMED_COLUMNS)
)

df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)
df.show(5)

### Vivo/Volte

In [ ]:
%%time
source_file = cdr_extraido_vivo_volte
target_file = cdr_transformado_vivo_volte
date_time_fmt = "yyyyMMdd HHmmss"

voice_mail_columns_to_keep = ["referencia", "data_hora", "numero_de_b_formatado", "chamada_caixa_postal"]
df_voice_mail = (
    spark.read.parquet(source_file)
    .filter(F.col("tipo_de_chamada") == "3")
    .withColumn(
            "numero_de_a_formatado",
            F.col("numero_de_a").substr(-11, 11)
        )
        .withColumn(
            "numero_de_b_formatado",
            F.col("numero_de_b").substr(-11, 11)
    )

    .filter(F.col("numero_de_a_formatado") == F.col("numero_de_b_formatado"))
    .withColumn("chamada_caixa_postal", F.lit(1))
)
df_voice_mail = _format_columns(df_voice_mail, date_time_fmt).select(voice_mail_columns_to_keep)
df_voice_mail.show(5)

In [ ]:
df = spark.read.parquet(source_file).filter(F.col("tipo_de_chamada") == "4")
df = _format_columns(df, date_time_fmt)
df = _format_numbers(df)
df = _add_chamada_curta(df)
df = df.join(
    df_voice_mail,
    on=["referencia", "data_hora", "numero_de_b_formatado"],
    how="left"
).withColumn(
    "chamada_caixa_postal", 
    F.coalesce(F.col("chamada_caixa_postal"), F.lit(0))
).withColumn("chamada_autenticada",F.lit(0)).select(TRANSFORMED_COLUMNS)

df.show(5)

In [ ]:
df_voice_mail.count()

In [ ]:
%%time
source_file = cdr_extraido_vivo_volte
target_file = cdr_transformado_vivo_volte

columns_to_keep = ["referencia", "data_hora", "numero_de_b_formatado", "caixa_postal"]
df_caixa_postal = (
    spark.read
        .parquet(source_file)
        .filter(F.col("tipo_de_chamada") == "3")
        .withColumn("data_hora",F.to_timestamp(F.concat_ws(" ",F.col("_data"), F.col("_hora")), "yyyyMMdd HHmmss"))
        .withColumn("numero_de_a", F.split(F.col("numero_de_a"), ";").getItem(0))
        # nas chamadas encaminhadas para caixa postal o número de b é acrescido de dois dígitos entre o código do país e o cn
        # neste caso considerar apenas os 11 números mais à direita de b para formatar corretamente 
        # pois não estão aderentes ao plano de numeração brasileiro
        .withColumn(
            "numero_de_b",
            F.col("numero_de_b").substr(-11, 11)
        )
        .withColumn(
            "numero_de_a_formatado",
            spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
        .withColumn( 
            "numero_de_b_formatado",
            spark_normalize_number("numero_de_b").getField("numero_formatado")
        )
        .filter(F.col("numero_de_a_formatado") == F.col("numero_de_b_formatado"))
        .withColumn("caixa_postal", F.lit(1))
        .select(columns_to_keep)
)

df = (
    spark.read
        .parquet(source_file)
        .filter(F.col("tipo_de_chamada") == "4")
        .withColumn("data_hora",F.to_timestamp(F.concat_ws(" ",F.col("_data"), F.col("_hora")), "yyyyMMdd HHmmss"))
        .withColumn("numero_de_a", F.split(F.col("numero_de_a"), ";").getItem(0))
        .withColumn(
            "numero_de_a_formatado",
            spark_normalize_number("numero_de_a").getField("numero_formatado")
        )
        .withColumn( 
            "numero_de_b_formatado",
            spark_normalize_number("numero_de_b").getField("numero_formatado")
        )
        .withColumn(
            "duracao_da_chamada", 
            F.coalesce(F.col("duracao_da_chamada").cast(T.IntegerType()), F.lit(0))
        )
        .withColumn("hora_da_chamada",F.date_format(F.col("data_hora"),"yyyyMMddHH"))
        .withColumn("chamada_autenticada", F.lit(0))
        .withColumn(
            "chamada_curta",
            F.when(F.col("duracao_da_chamada") <= LIMIAR_CHAMADA_OFENSORA, 1).otherwise(F.lit(0))
        )
        .join(
            F.broadcast(df_caixa_postal),
            on=["referencia", "data_hora", "numero_de_b_formatado"], 
            how="left"
        )
        .fillna(0, subset=["caixa_postal"])
        .select(TRANSFORMED_COLUMNS)
)

df.write.mode("overwrite").partitionBy("tipo_de_chamada").parquet(target_file)
df.show(10)

## Carga e cálculo dos ofensores